# Lab 02: Matrix Operations และการแก้ระบบสมการเชิงเส้น
> CLO1 | LLo: ดำเนินการทางแมทริกซ์ แก้ระบบสมการ Ax = b ด้วย Elimination และคำนวณ Determinant ได้

**วิชา**: 1145 201 คณิตศาสตร์สำหรับวิทยาการข้อมูล | **Strang Reference**: Ch.2, Ch.5

---

## บทนำ

สัปดาห์นี้เราจะเรียนรู้เรื่อง **Matrix** ซึ่งเป็นการขยาย Vector ให้เป็นสองมิติ Matrix ไม่ใช่แค่ตารางตัวเลข แต่คือเครื่องมือในการแทน **transformation** ของข้อมูล และการแก้ **ระบบสมการเชิงเส้น** ซึ่งเป็นพื้นฐานของ Machine Learning ทุกอย่าง เป้าหมายของ Lab นี้คือให้นักศึกษาสามารถสร้างและดำเนินการกับ Matrix ด้วย NumPy ได้ รวมถึงแก้ระบบสมการ Ax = b ด้วยทั้ง NumPy และการ implement Gaussian Elimination ด้วยตัวเอง ใน Data Science ทุกครั้งที่เราฝึกโมเดล Linear Regression หรือ Neural Network ระบบกำลังแก้สมการ Matrix อยู่เบื้องหลัง Lab นี้มี 3 TODOs: สร้าง Matrix, Gaussian Elimination from scratch, และ Case Study การวิเคราะห์ข้อมูลราคาบ้าน

In [ ]:
# ─── Import libraries ────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ที่จำเป็นสำหรับ Matrix operations
# numpy: สำหรับ matrix creation และ operations
# scipy.linalg: สำหรับ LU decomposition
# matplotlib: สำหรับ visualization

import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

print('Libraries loaded successfully!')
print('NumPy:', np.__version__)

## Part 1: Matrix Types และ Basic Operations

**Part นี้เราจะสร้างและดำเนินการกับ Matrix พื้นฐาน เพื่อให้เชื่อมโยง math notation กับ NumPy code ได้**

Matrix **A** ∈ ℝᵐˣⁿ คือตารางตัวเลข m แถว n คอลัมน์ ใน Data Science: X ∈ ℝⁿˣᵖ แทน dataset ที่มี n samples และ p features — นี่คือ Matrix ที่เราจะเห็นบ่อยที่สุดตลอดวิชา

In [ ]:
# ─── Demo: สร้าง Matrix รูปแบบต่าง ๆ ─────────────────────────────
# วัตถุประสงค์: แสดง matrix types ที่สำคัญใน Linear Algebra

# Matrix จาก nested list
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print('=== Matrix A (3x3) ===')
print(A)
print('shape:', A.shape)  # (3, 3)

# Identity matrix
I = np.eye(3)
print('\n=== Identity Matrix I3 ===')
print(I)

# Transpose
print('\n=== Transpose A.T ===')
print(A.T)

# Symmetric check: A = A^T?
B = np.array([[1, 2, 3],
              [2, 5, 6],
              [3, 6, 9]])
print('\nIs B symmetric?', np.allclose(B, B.T))  # True

In [ ]:
# ─── Demo: Matrix Operations ──────────────────────────────────────
# วัตถุประสงค์: แสดง add, scalar multiply, matrix multiply และ verify property

A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])

print('A =\n', A)
print('B =\n', B)

# Addition
print('\nA + B =\n', A + B)

# Scalar multiply
print('\n2 * A =\n', 2 * A)

# Matrix multiply — @ operator หรือ np.matmul()
print('\nA @ B =\n', A @ B)
print('B @ A =\n', B @ A)
print('AB == BA?', np.allclose(A @ B, B @ A))  # False: non-commutative

# (AB)^T = B^T A^T
lhs = (A @ B).T
rhs = B.T @ A.T
print('\n(AB)^T == B^T A^T?', np.allclose(lhs, rhs))  # True

### 🎯 TODO 1: สร้าง Dataset Matrix และวิเคราะห์ (ระดับ: Easy)

เราต้องการแทน **Dataset ของนักศึกษา** เป็น Matrix **X** ∈ ℝ⁵ˣ³ เพื่อแสดงให้เห็นว่า Machine Learning มองข้อมูลเป็น Matrix ได้อย่างไร — n=5 samples, p=3 features

| นักศึกษา | Math (x₁) | Python (x₂) | Study hrs (x₃) |
|---------|-----------|------------|---------------|
| A | 82 | 88 | 15 |
| B | 75 | 70 | 10 |
| C | 91 | 95 | 20 |
| D | 60 | 65 | 8 |
| E | 88 | 80 | 18 |

ให้คุณ:
1. สร้าง numpy array **X** จากข้อมูลด้านบน (dtype=float64)
2. Print: shape, mean ของแต่ละ column (`X.mean(axis=0)`), std ของแต่ละ column
3. Standardize **X** → **Z** โดย Z = (X - μ) / σ (เช่น mean=0, std=1 ต่อ column)
4. Verify ว่า Z.mean(axis=0) ≈ [0, 0, 0] และ Z.std(axis=0) ≈ [1, 1, 1]

In [ ]:
# TODO 1: เขียน code ที่นี่
# Hint: สร้าง X ด้วย np.array([[82, 88, 15], ...])
# Hint: X.mean(axis=0) คำนวณ mean ตาม column (axis=0 = แนวตั้ง)
# Hint: Z = (X - X.mean(axis=0)) / X.std(axis=0)

raise NotImplementedError('กรุณาเติม code ใน TODO 1')

## Part 2: Solving Linear Systems — Ax = b

**Part นี้เราจะแก้ระบบสมการเชิงเส้นด้วย 3 วิธี เพื่อเปรียบเทียบและเข้าใจว่าแต่ละวิธีมี trade-off อย่างไร**

ปัญหาพื้นฐาน: กำหนด **A** และ **b** หา **x** ที่ทำให้ **Ax = b**

ใน Data Science: Normal Equations ของ Linear Regression คือ **(XᵀX)x = Xᵀy** — ซึ่งก็คือ Ax = b นั่นเอง!

In [ ]:
# ─── Demo: แก้ Ax = b สามวิธี ──────────────────────────────────────
# วัตถุประสงค์: เปรียบเทียบ np.linalg.solve, inverse matrix, LU decomposition
# เพื่อเข้าใจว่า NumPy ใช้ LU internally สำหรับ solve()

# ระบบสมการ:
# 2x + y - z = 8
# -3x - y + 2z = -11
# -2x + y + 2z = -3

A = np.array([[ 2,  1, -1],
              [-3, -1,  2],
              [-2,  1,  2]], dtype=float)
b = np.array([8, -11, -3], dtype=float)

print('=== วิธีที่ 1: np.linalg.solve (แนะนำ) ===')
x1 = np.linalg.solve(A, b)
print('x =', x1)  # [2, 3, -1]
print('Verify Ax = b:', np.allclose(A @ x1, b))

print('\n=== วิธีที่ 2: Inverse matrix (ช้ากว่า) ===')
x2 = np.linalg.inv(A) @ b
print('x =', x2)

print('\n=== วิธีที่ 3: LU Decomposition (scipy) ===')
P, L, U = linalg.lu(A)
print('P =\n', P)
print('L =\n', L)
print('U =\n', U)
print('Verify PA = LU:', np.allclose(P @ A, L @ U))

# Determinant — ถ้า det ≠ 0 แสดงว่า unique solution มี
print('\ndet(A) =', np.linalg.det(A))

### 🎯 TODO 2: Gaussian Elimination From Scratch (ระดับ: Medium)

เราต้องการ implement **Gaussian Elimination** ด้วยตัวเองเพื่อเข้าใจ algorithm ที่ NumPy ใช้อยู่เบื้องหลัง — ทักษะนี้ช่วยให้เข้าใจ numerical stability ของ solver ต่าง ๆ ซึ่งสำคัญมากใน large-scale ML

แก้ระบบสมการ:
```
x₁ + 2x₂ - x₃ = 2
2x₁ + x₂ + x₃ = 5  
-x₁ + 3x₂ + 2x₃ = 4
```

ให้คุณ implement function `gaussian_elimination(A, b)` ที่:
1. สร้าง augmented matrix [A|b]
2. Forward elimination: ทำให้ elements ใต้ diagonal เป็น 0
3. Back substitution: หาค่า x จาก upper triangular matrix
4. Return x

แล้ว verify ว่าผลเท่ากับ `np.linalg.solve(A, b)`

In [ ]:
# TODO 2: เขียน Gaussian Elimination from scratch
# Hint: augmented = np.column_stack([A.copy(), b.copy()])
# Hint: forward elimination: สำหรับแต่ละ pivot row i:
#         สำหรับแต่ละ row j > i: multiplier = aug[j,i] / aug[i,i]
#         aug[j] -= multiplier * aug[i]
# Hint: back substitution: x[n-1] = aug[n-1, n] / aug[n-1, n-1]
#         ย้อนจาก i = n-2 ถึง 0: x[i] = (aug[i,n] - sum(aug[i,i+1:n] * x[i+1:])) / aug[i,i]

def gaussian_elimination(A, b):
    """แก้ระบบสมการ Ax = b ด้วย Gaussian Elimination"""
    # TODO: implement ที่นี่
    raise NotImplementedError('กรุณาเติม code ใน TODO 2')

# ทดสอบ
A = np.array([[ 1,  2, -1],
              [ 2,  1,  1],
              [-1,  3,  2]], dtype=float)
b = np.array([2, 5, 4], dtype=float)

x_elim = gaussian_elimination(A, b)
x_numpy = np.linalg.solve(A, b)
print('Gaussian elimination:', x_elim)
print('np.linalg.solve:     ', x_numpy)
print('ผลเท่ากัน?', np.allclose(x_elim, x_numpy))

## Part 3: Determinant และ Singular Matrix

**Part นี้เราจะสำรวจ Determinant และเข้าใจความสัมพันธ์กับ Invertibility เพราะ det = 0 หมายถึง singular matrix ซึ่งใน DS เชื่อมกับ multicollinearity**

In [ ]:
# ─── Demo: Determinant และ Singular Matrix ──────────────────────────
# วัตถุประสงค์: แสดงว่า det = 0 เชื่อมกับ singular matrix และ no solution อย่างไร

# Non-singular matrix
A = np.array([[2, 1], [1, 3]], dtype=float)
print('=== Non-singular A ===')
print('A =\n', A)
print('det(A) =', np.linalg.det(A))  # 5.0
print('inv(A) =\n', np.linalg.inv(A))
print('A @ inv(A) =\n', A @ np.linalg.inv(A))

# Singular matrix (rows linearly dependent)
B = np.array([[1, 2], [2, 4]], dtype=float)  # row 2 = 2 × row 1
print('\n=== Singular B ===')
print('B =\n', B)
print('det(B) =', np.linalg.det(B))  # ≈ 0
try:
    inv_B = np.linalg.inv(B)
    print('inv(B) =', inv_B)
except np.linalg.LinAlgError as e:
    print('Error:', e)  # Singular matrix

# Rank
print('\nrank(A) =', np.linalg.matrix_rank(A))  # 2 (full rank)
print('rank(B) =', np.linalg.matrix_rank(B))  # 1 (rank deficient)

### 🎯 TODO 3: Case Study — แก้ระบบสมการราคาบ้าน (ระดับ: Hard)

บริษัทอสังหาริมทรัพย์มีสมการพยากรณ์ราคาบ้าน (หน่วย: ล้านบาท):

```
β₀ + 150β₁ + 3β₂ + 8β₃ = 3.2    (บ้านหลัง 1: 150ตรม, 3ห้อง, อายุ8ปี, ราคา3.2M)
β₀ + 200β₁ + 4β₂ + 5β₃ = 4.8    (บ้านหลัง 2: 200ตรม, 4ห้อง, อายุ5ปี, ราคา4.8M)
β₀ + 120β₁ + 2β₂ + 15β₃ = 2.1   (บ้านหลัง 3: 120ตรม, 2ห้อง, อายุ15ปี, ราคา2.1M)
β₀ + 180β₁ + 3β₂ + 3β₃ = 4.0    (บ้านหลัง 4: 180ตรม, 3ห้อง, อายุ3ปี, ราคา4.0M)
```

ให้คุณ:
1. สร้าง matrix A และ vector b จากระบบสมการด้านบน
2. แก้หา β = [β₀, β₁, β₂, β₃] ด้วย `np.linalg.solve()`
3. ตรวจสอบ det(A) — ถ้า det ≈ 0 แสดงว่า multicollinearity มีปัญหา
4. Verify: คำนวณ A @ β และเปรียบเทียบกับ b
5. ทำนายราคาบ้านหลังใหม่: 160ตรม, 3ห้อง, อายุ10ปี
6. **ตีความ**: β₁ หมายความว่าอะไร? พื้นที่เพิ่ม 1 ตรม ราคาเปลี่ยนเท่าไร?

In [ ]:
# TODO 3: Case Study — House Price Linear System
# Hint: A คือ design matrix ที่ column แรกเป็น 1 ทั้งหมด (สำหรับ β₀)
# Hint: A = [[1, 150, 3, 8], [1, 200, 4, 5], [1, 120, 2, 15], [1, 180, 3, 3]]
# Hint: b = [3.2, 4.8, 2.1, 4.0]
# Hint: ทำนาย: np.dot([1, 160, 3, 10], beta)

raise NotImplementedError('กรุณาเติม code ใน TODO 3')

## Reflection Questions

**คำถามที่ 1**: ทำไม `np.linalg.solve(A, b)` ดีกว่า `np.linalg.inv(A) @ b` ในแง่ของ numerical stability? (Hint: ค้นหา condition number)

*(เขียนคำตอบที่นี่)*

---

**คำถามที่ 2**: ถ้า dataset ของเรามี feature สองตัวที่เป็น linear combination กัน (เช่น x₂ = 2x₁) จะเกิดอะไรขึ้นกับ det(XᵀX)? และส่งผลต่อ Model อย่างไร?

*(เขียนคำตอบที่นี่)*

---

**คำถามที่ 3**: ใน TODO 3 เราแก้ square system (4 equations, 4 unknowns) แต่ Dataset จริงมักมี n >> p (เช่น 1000 samples, 4 features) ซึ่งเรียกว่า overdetermined system ทำไม np.linalg.solve จึงใช้ไม่ได้ และต้องใช้ np.linalg.lstsq แทน?

*(เขียนคำตอบที่นี่)*